# S05 - Time-series momentum (trend)

**Hypothesis.** Asset returns are positively autocorrelated at 3-12 month horizons
(Moskowitz, Ooi & Pedersen 2012). Holding the S&P 500 only when its own trailing excess
return is positive captures this and sidesteps long bear markets, with a different
lookback structure than the price-vs-SMA200 rule.

| Variant | Risk-on (hold S&P 500) when ... |
|---|---|
| m12_skip1 | total return from t-12m to t-1m > T-bill accrual over the same window |
| m12 | total return over the last 12 months > T-bill accrual |
| m6 | total return over the last 6 months > T-bill accrual |

**Inputs and point-in-time lags**

| Series | Source | Lag used |
|---|---|---|
| S&P 500 daily total return, T-bill daily return | `bt_long.long_sample` | up to the decision close; 21 trading days per month; next-close fill |

Monthly check, next-close execution, 5 bp per switch.

**Sources.** Moskowitz, Ooi & Pedersen (2012), *Time series momentum*, JFE, SSRN 2089463;
Hurst, Ooi & Pedersen (2017), *A century of evidence on trend-following investing*, SSRN 2993026;
Zakamulin (2015), *Market timing with moving averages*, SSRN 2585056.

> Decision-support research only; never places orders.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
import bt_common as bc  # noqa: E402
import bt_long  # noqa: E402
import bt_periods as bp  # noqa: E402
import bt_pit  # noqa: E402
import bt_registry as br  # noqa: E402
import bt_stats as bs  # noqa: E402
import seed_registry  # noqa: E402
import stage2 as s2  # noqa: E402

bc.setup_plotly("cdn")
pd.set_option("display.width", 180)
TRIAL_ID, STRATEGY, NOTEBOOK = 'S05', 'tsmom', 'trend/tsmom.ipynb'
PROTOCOL, REBALANCE = 'main', 'monthly'

sample = bt_long.long_sample()
px, cash = sample["px"], sample["cash"]
bench = bp.benchmark_returns(px, cash)  # SMA200 monthly / B&H / 60-40 / T-bills, frozen settings
print("prices", px.index[0].date(), "->", px.index[-1].date(), "| registry trials so far:", br.trial_count())

prices 1934-02-01 -> 2026-10-19 | registry trials so far: 17


## 1. Data and point-in-time inputs

In [2]:
inputs = {"px": px, "cash": cash}
print("daily TR and T-bill returns", px.index[0].date(), "->", px.index[-1].date())

daily TR and T-bill returns 1934-02-01 -> 2026-10-19


## 2. Signal and variant grid (fixed before any backtest)

In [3]:
M = 21  # trading days per month


def build_signal(inputs, look=12, skip=0):
    tr = inputs["px"]["tr_close"]
    ci = (1 + inputs["cash"].reindex(tr.index).fillna(0.0)).cumprod()
    a, b = look * M, skip * M
    excess = (tr.shift(b) / tr.shift(a) - 1) - (ci.shift(b) / ci.shift(a) - 1)
    return (excess > 0).astype(float).where(excess.notna())


VARIANTS = {"m12_skip1": dict(look=12, skip=1), "m12": dict(look=12), "m6": dict(look=6)}
PARAMS_TEXT = {n: f"{n}; trailing TR minus T-bill accrual > 0; monthly; next_close; 5bp" for n in VARIANTS}

## 3. Look-ahead checks

Every variant is recomputed with all inputs cut at past dates (monthly inputs: completed months only) and must equal the full-data signal on those dates (`stage2.truncation_test`). No backtest runs unless all pass.

In [4]:
N_CUTS = 30
trunc = {}
for name, params in VARIANTS.items():
    trunc[name] = s2.truncation_test(build_signal, inputs, n_cuts=N_CUTS, **params)
display(pd.DataFrame({k: v.row() for k, v in trunc.items()}).T)
bad = [k for k, v in trunc.items() if not v.passed]
assert not bad, f"look-ahead detected in variants {bad}: fix before any backtest"
print("all variants pass the truncation look-ahead test")

,cuts,mismatches,max_abs_diff,passed
m12_skip1,30,0,0.0,True
m12,30,0,0.0,True
m6,30,0,0.0,True


all variants pass the truncation look-ahead test


## 4. Development-period sweep

Each variant is scored on the **dev** period only and registered as a trial. The best dev Sharpe is chosen; validation and holdout results are never used for the choice.

In [5]:
signals = {name: build_signal(inputs, **params) for name, params in VARIANTS.items()}
sweep, CHOSEN = s2.dev_sweep(px, cash, bench, signals, PARAMS_TEXT, trial_id=TRIAL_ID, strategy=STRATEGY,
                             notebook=NOTEBOOK, protocol=PROTOCOL, rebalance=REBALANCE)
display(sweep)
print("chosen on dev Sharpe only:", CHOSEN, "|", PARAMS_TEXT[CHOSEN])

,dev_sharpe,d_sma200,d_bh,time_in_market,switches_pa,dev_start
m12_skip1,0.415547,-0.213798,-0.083995,0.670659,1.187926,1936-02-13
m12,0.443663,-0.185682,-0.055878,0.673542,0.96519,1936-02-13
m6,0.651151,0.007415,0.140152,0.658097,1.471077,1935-08-12


chosen on dev Sharpe only: m6 | m6; trailing TR minus T-bill accrual > 0; monthly; next_close; 5bp


## 5. Frozen evaluation of the chosen variant

Frozen primary metric (`PLAN.md` section 2): beat SMA200 and buy & hold in dev **and** val, one-sided block-bootstrap p <= 0.10 vs SMA200 in both, deflated Sharpe >= 0.95 with the registry trial count. Holdout is confirmatory only.

In [6]:
signal = signals[CHOSEN]
runs, cmp, stats, res = s2.evaluate(px, cash, bench, signal, protocol=PROTOCOL, rebalance=REBALANCE)
display(cmp.round(3))
display(stats)
display(res.to_frame().round(3))
print(f"n_trials={res.n_trials} | DSR={res.dsr:.3f} | passes={res.passes} | holdout confirms={res.holdout_confirms}")
display(pd.Series(res.checks, name="check"))

,strategy,bh,d_bh,sma200,d_sma200,mix60,d_mix60
dev,0.651,0.511,0.140,0.644,0.007,0.518,0.133
val,0.487,0.319,0.168,0.572,-0.085,0.309,0.178
holdout,0.458,0.781,-0.322,0.658,-0.199,0.779,-0.321
full,0.573,0.513,0.059,0.630,-0.057,0.515,0.058


,CAGR,Vol,Sharpe,Sortino,Max DD,Calmar,Worst day,MaxDD peak,MaxDD recovered,Final x,Time in market,Switches,Switches / yr,Cost drag / yr
dev,0.111615,0.113903,0.651151,0.894907,-0.4172,0.267533,-0.204547,1938-10-10,1944-12-08,315.562708,0.658097,80,1.471077,0.000736
val,0.090037,0.115004,0.486884,0.687517,-0.191858,0.469291,-0.068651,1998-07-17,1999-07-01,5.605593,0.660916,34,1.700466,0.00085
holdout,0.071086,0.139723,0.458266,0.623744,-0.337905,0.210372,-0.119799,2020-02-19,2021-08-11,3.167409,0.780777,26,1.548679,0.000774
full,0.099276,0.119313,0.572519,0.78911,-0.4172,0.237959,-0.204547,1938-10-10,1944-12-08,5602.880522,0.681326,140,1.535309,0.000768


,Sharpe,d_sma200,d_bh,p_vs_sma200
dev,0.651,0.007,0.140,0.547
val,0.487,-0.085,0.168,0.938
holdout,0.458,-0.199,-0.322,NaN
full,0.573,-0.057,0.059,NaN


n_trials=20 | DSR=1.000 | passes=False | holdout confirms=False


dev_beats_sma200         True
dev_beats_bh             True
dev_p_le_0.1            False
val_beats_sma200        False
val_beats_bh             True
val_p_le_0.1            False
dsr_ge_0.95              True
holdout_d_sma200_pos    False
Name: check, dtype: bool

In [7]:
s2.plot_growth(runs, bench, f'{TRIAL_ID} {CHOSEN}', PROTOCOL).show()

## 6. Robustness: costs, placebo, peek test, decades

In [8]:
cs = bs.cost_sensitivity(px, signal, cash, rebalance=REBALANCE)
display(cs.round(3))
sma_full = bs.excess_sharpe(bench["sma200"].reindex(runs["full"].index), bench["tbill"].reindex(runs["full"].index))
be = bs.breakeven_cost_bps(px, signal, cash, bench_sharpe=sma_full, rebalance=REBALANCE)
print(f"break-even one-way cost vs SMA200 Sharpe over the same window ({sma_full:.3f}): {be:.1f} bp")
pl = bs.placebo_test(px, signal, cash, n_placebo=200, rebalance=REBALANCE, rng=np.random.default_rng(bs.SEED))
display(pd.Series(pl, name="placebo").to_frame())
peek = bt_pit.lookahead_test(px, signal, cash, rebalance=REBALANCE)
print(f"peek test: Sharpe by shift {peek.sharpe_by_shift.round(3).to_dict()} | gain {peek.peek_gain:+.3f}")
display(s2.decade_table(runs["full"], bench).style.format("{:.2f}"))

,Sharpe,CAGR,Max DD,turnover_pa
cost_bps,,,,
0,0.590,0.101,-0.415,1.555
5,0.584,0.101,-0.417,1.555
10,0.577,0.100,-0.420,1.555
25,0.558,0.097,-0.426,1.555
50,0.525,0.093,-0.438,1.555


break-even one-way cost vs SMA200 Sharpe over the same window (0.630): 0.0 bp


,placebo
stat,Sharpe
real,0.583724
placebo_mean,0.422024
placebo_q95,0.554851
quantile,0.995
p_one_sided,0.005
n_placebo,200
time_in_market,0.67897


peek test: Sharpe by shift {-21: 1.257, -5: 0.708, -1: 0.589, 0: 0.584, 1: 0.585, 5: 0.555, 21: 0.57} | gain +0.673


,Sharpe,Sharpe B&H,Sharpe SMA200,CAGR,CAGR B&H,CAGR SMA200,time in market
1930s,0.40,0.26,0.41,0.06,0.03,0.06,0.67
1940s,0.82,0.61,0.57,0.09,0.09,0.06,0.67
1950s,1.39,1.42,1.51,0.17,0.19,0.18,0.81
1960s,0.55,0.40,0.43,0.08,0.08,0.07,0.64
1970s,0.23,0.03,0.37,0.08,0.06,0.09,0.48
1980s,0.56,0.53,0.58,0.17,0.17,0.17,0.69
1990s,0.46,0.92,0.64,0.10,0.18,0.13,0.78
2000s,0.52,-0.05,0.50,0.08,-0.01,0.08,0.54
2010s,0.58,0.90,0.71,0.07,0.14,0.09,0.80
2020s,0.33,0.67,0.58,0.07,0.15,0.10,0.75


## 7. Register the result

In [9]:
note = ("passes" if res.passes else "fails") + f" frozen metric; chosen {CHOSEN} on dev; protocol {PROTOCOL}"
rows = s2.register_final(TRIAL_ID, STRATEGY, NOTEBOOK, PARAMS_TEXT[CHOSEN], res, note=note, header=seed_registry.HEADER)
display(rows)
print("registry trial count:", br.trial_count())

,trial_id,date,strategy,notebook,params,period,sharpe,d_sma200,d_bh,p_vs_sma200,dsr,n_trials,passes,status,note
49,S05-v1/dev,2026-10-20,tsmom,trend/tsmom.ipynb,m12_skip1; trailing TR minus T-bill accrual > ...,dev,0.4155,-0.2138,-0.0840,,,,,run,"dev-only sweep, variant m12_skip1"
50,S05-v2/dev,2026-10-20,tsmom,trend/tsmom.ipynb,m12; trailing TR minus T-bill accrual > 0; mon...,dev,0.4437,-0.1857,-0.0559,,,,,run,"dev-only sweep, variant m12"
51,S05-v3/dev,2026-10-20,tsmom,trend/tsmom.ipynb,m6; trailing TR minus T-bill accrual > 0; mont...,dev,0.6512,0.0074,0.1402,,,,,run,"dev-only sweep, variant m6"
52,S05/dev,2026-10-20,tsmom,trend/tsmom.ipynb,m6; trailing TR minus T-bill accrual > 0; mont...,dev,0.6512,0.0074,0.1402,0.5466,,20,False,fail,fails frozen metric; chosen m6 on dev; protoco...
53,S05/val,2026-10-20,tsmom,trend/tsmom.ipynb,m6; trailing TR minus T-bill accrual > 0; mont...,val,0.4869,-0.0853,0.1678,0.9382,,20,False,fail,fails frozen metric; chosen m6 on dev; protoco...
54,S05/holdout,2026-10-20,tsmom,trend/tsmom.ipynb,m6; trailing TR minus T-bill accrual > 0; mont...,holdout,0.4583,-0.1993,-0.3225,,,20,False,fail,fails frozen metric; chosen m6 on dev; protoco...
55,S05/full,2026-10-20,tsmom,trend/tsmom.ipynb,m6; trailing TR minus T-bill accrual > 0; mont...,full,0.5725,-0.0572,0.0592,,0.9996,20,False,fail,fails frozen metric; chosen m6 on dev; protoco...


registry trial count: 20


## 8. Current signal

In [10]:
last = signal.dropna()
print(f"latest signal ({CHOSEN}) on {last.index[-1].date()}: equity weight {last.iloc[-1]:.2f}")
chk = bc.check_days(px.index, REBALANCE)
display(last[chk.reindex(last.index).fillna(False)].tail(6).to_frame("equity weight at decision"))
print("Research output only -- not a recommendation or an order.")

latest signal (m6) on 2026-10-19: equity weight 1.00


,equity weight at decision
Date,
2026-04-30,1.0
2026-05-29,1.0
2026-06-30,1.0
2026-07-31,1.0
2026-08-31,1.0
2026-09-30,1.0


Research output only -- not a recommendation or an order.


## 9. Evidence summary (computed)

In [11]:
for line in s2.evidence_lines(res, trunc[CHOSEN], peek, pl, be):
    print("*", line)

* Truncation look-ahead test: PASSED on 30 cut dates; peek test gain +0.673.
* dev: Sharpe 0.651 (vs SMA200 +0.007, vs B&H +0.140, p vs SMA200 0.547)
* val: Sharpe 0.487 (vs SMA200 -0.085, vs B&H +0.168, p vs SMA200 0.938)
* holdout: Sharpe 0.458 (vs SMA200 -0.199, vs B&H -0.322)
* Deflated Sharpe 1.000 with n_trials=20.
* Placebo: real 0.584 vs placebo 95th pct 0.555 (p 0.005); break-even cost vs SMA200 full 0.0 bp.
* Verdict: FAILS the frozen primary metric (failed checks: dev_p_le_0.1, val_beats_sma200, val_p_le_0.1); holdout does not confirm.


## 10. Caveats

* Closely related to SMA200 (both are trend rules on the same index), so a large Sharpe difference
  between them is not expected; the bootstrap test against SMA200 is demanding here.
* 21 trading days per month is an approximation of calendar months.
* Single asset: the diversification benefit behind published TSMOM results (many markets) is absent.

## 11. Interpretation

* The dev sweep preferred 6-month momentum (Sharpe 0.65) over the textbook 12-1 (0.42) and 12-month
  (0.44) lookbacks. It essentially **ties SMA200 in dev** (+0.007, p 0.55), trails it in val (-0.09, p 0.94)
  and is clearly worse in the holdout (0.46 vs 0.66). **Fails** the frozen metric.
* The placebo (p 0.005) and the peek test (+0.67 Sharpe from peeking 21 days) show the signal carries
  real timing information, as expected for a trend rule. But it is the *same* information SMA200
  already uses: price trend on the same index. It adds no edge over the simpler benchmark.
* The faster 6-month lookback whipsaws in V-shaped recoveries (2010s, 2020s), which is why it loses
  most in the holdout; SMA200 with monthly checks is slower and more robust there.
* That the classic 12-1 variant is the weakest on this single index is a reminder that published
  TSMOM gains come largely from diversification across many markets.